# PANDAS MASTERY — CHAPTERS 19–24

```
PANDAS MASTERY
Current Chapters: 19 (Vectorization), 20 (Conditional Column Creation), 21 (GroupBy),
                  22 (GroupBy Advanced), 23 (MultiIndex), 24 (Merge)
Progress:         24 / 76 chapters
Completed:        Ch1-18
Remaining:        52 chapters
```

Six chapters, the heaviest yet. GroupBy and Merge are arguably the two most important skills in
this entire course — nearly every real analytical question ("revenue by region," "customers
with their orders") reduces to one or both. MultiIndex is the natural structure `groupby`
produces with multiple keys, so it sits right before Merge, where you'll need to `reset_index()`
grouped results back to flat columns before joining them to anything else.


In [ ]:
import pandas as pd
import numpy as np
print("pandas version:", pd.__version__)


---
# CHAPTER 19 — Vectorization

## Concept
Chapter 18 already measured this decisively, so this chapter formalizes the *why* and extends
it to boolean/string operations specifically, and to NumPy interoperability.


In [ ]:
import timeit

n = 200_000
df = pd.DataFrame({
    "price": np.random.default_rng(0).uniform(10, 500, n),
    "quantity": np.random.default_rng(1).integers(1, 20, n),
})

def loop_total():
    totals = []
    for i in range(len(df)):
        totals.append(df["price"].iloc[i] * df["quantity"].iloc[i])
    return totals

def vectorized_total():
    return df["price"] * df["quantity"]

t_loop = timeit.timeit(loop_total, number=1)
t_vec = timeit.timeit(vectorized_total, number=1)
print(f"Loop with .iloc per row: {t_loop:.4f}s")
print(f"Vectorized (price*qty) : {t_vec:.4f}s   ({t_loop/t_vec:.0f}x faster)")

# --- Boolean vectorization ---
t_loop_bool = timeit.timeit(lambda: [p > 250 for p in df["price"]], number=1)
t_vec_bool = timeit.timeit(lambda: df["price"] > 250, number=1)
print(f"\nBoolean loop: {t_loop_bool:.4f}s   Boolean vectorized: {t_vec_bool:.4f}s   ({t_loop_bool/t_vec_bool:.0f}x faster)")


### Expected Output (verified on pandas 3.0.2 — exact numbers vary by machine, order is the point)
```
Loop with .iloc per row: 11.4297s
Vectorized (price*qty) : 0.0012s   (9569x faster)

Boolean loop: 0.0242s   Boolean vectorized: 0.0005s   (45x faster)
```

### Why the `.iloc`-per-row loop is catastrophically slower than Chapter 18's `.apply()` case
This loop calls `.iloc[i]` **twice per row** — each `.iloc` call pays its own general-purpose
dispatch overhead (Chapter 6), on top of the per-element Python loop overhead itself. This is a
genuinely common real-world anti-pattern: it "looks like" normal Python code, gives no error,
and is often the very first thing someone new to Pandas writes when they want "the price times
quantity for each row" — and it's routinely thousands of times slower than the one-line
vectorized equivalent. **If you ever see `.iloc[i]` or `.loc[label]` inside a `for` loop over
DataFrame rows, that's almost always a sign the operation should be vectorized instead.**

### NumPy interoperability — operations transfer directly


In [ ]:
s = pd.Series([1.0, 4.0, 9.0, 16.0, 25.0])

# NumPy ufuncs work DIRECTLY on Series -- no .values or .to_numpy() conversion needed
print("np.sqrt(s):", np.sqrt(s).tolist(), " <- returns a Series, index preserved")
print("np.where(s > 10, 'high', 'low'):", np.where(s > 10, "high", "low"), " <- returns a plain ndarray")

# Combining Pandas boolean masks with NumPy's np.select for multi-condition vectorized logic
conditions = [s < 5, s < 15, s >= 15]
choices = ["small", "medium", "large"]
categorized = np.select(conditions, choices, default="unknown")
print("\nnp.select() result:", categorized)


### Expected Output (verified on pandas 3.0.2)
```
np.sqrt(s): [1.0, 2.0, 3.0, 4.0, 5.0]  <- returns a Series, index preserved
np.where(s > 10, 'high', 'low'): ['low' 'low' 'low' 'high' 'high']  <- returns a plain ndarray

np.select() result: ['small' 'medium' 'medium' 'large' 'large']
```

### A real, current gotcha: `np.select` now requires a dtype-compatible `default`
Calling `np.select(conditions, choices)` **without** an explicit `default` raises a `TypeError`
on this NumPy version — newer NumPy no longer silently falls back to its old default of `0`
(an `int`) when the `choices` are strings, since it can no longer find a common dtype between
`int` and `str` to build the result array. **Always pass an explicit, dtype-compatible
`default=` to `np.select`** — this was caught by actually running the code, not by memory of
older NumPy behavior, and is exactly the kind of environment/version-dependent surprise this
notebook's "run everything, trust nothing from memory" discipline exists to catch.

### Interview Insight
> "You see a colleague's code using `.iloc[i]` inside a `for i in range(len(df))` loop to
> compute a new column. What would you tell them?"
This is almost always vectorizable — direct column arithmetic (`df['a'] * df['b']`) processes
the entire column at once via compiled operations, while the loop pays Python-level overhead
plus `.iloc`'s own dispatch overhead on every single access, often making it thousands of times
slower for large data. The fix is nearly always a one-line vectorized expression replacing the
entire loop.


---
# CHAPTER 20 — Conditional Column Creation

## Concept
`np.where()` for binary conditions, `np.select()` for multi-way conditions, `Series.where()`/
`Series.mask()` for Pandas-native conditional replacement — four related tools for "create a
column based on a condition," each suited to a different shape of logic.


In [ ]:
customers = pd.DataFrame({"name": ["Alice", "Bob", "Carol", "Dave", "Eve"], "age": [17, 25, 68, 42, 8]})

# --- np.where: simple binary condition ---
customers["is_adult"] = np.where(customers["age"] >= 18, "Adult", "Minor")
print("np.where (binary):\n", customers)

# --- np.select: multi-way condition, evaluated in ORDER, first match wins ---
conditions = [
    customers["age"] < 13,
    customers["age"] < 18,
    customers["age"] < 65,
]
choices = ["Child", "Teen", "Adult"]
customers["age_group"] = np.select(conditions, choices, default="Senior")
print("\nnp.select (multi-way):\n", customers)


### Expected Output (verified on pandas 3.0.2)
```
np.where (binary):
     name  age is_adult
0   Alice   17    Minor
1     Bob   25    Adult
2   Carol   68    Adult
3    Dave   42    Adult
4     Eve    8    Minor

np.select (multi-way):
     name  age is_adult age_group
0   Alice   17    Minor      Teen
1     Bob   25    Adult     Adult
2   Carol   68    Adult    Senior
3    Dave   42    Adult     Adult
4     Eve    8    Minor     Child


### `Series.where()` and `Series.mask()` — the inverse-logic pair
`where(cond, other)` keeps values where `cond` is `True`, replaces with `other` where `False`.
`mask(cond, other)` does the **opposite** — replaces where `cond` is `True`, keeps where
`False`. Easy to mix up; the names are the mnemonic: "where [this holds], [keep it]" vs
"[apply a] mask [over the matching parts]."


In [ ]:
scores = pd.Series([45, 78, 92, 33, 67])

capped_where = scores.where(scores <= 50, other=50)     # keep where <=50, else replace with 50 -- a CAP
floored_mask = scores.mask(scores < 50, other=50)         # replace where <50 with 50, else keep -- a FLOOR

print("Original         :", scores.tolist())
print("where(<=50, 50)  :", capped_where.tolist(), " <- caps values ABOVE 50 down to 50")
print("mask(<50, 50)    :", floored_mask.tolist(), " <- floors values BELOW 50 up to 50")


### Expected Output (verified on pandas 3.0.2)
```
Original         : [45, 78, 92, 33, 67]
where(<=50, 50)  : [45, 50, 50, 33, 67]  <- caps values ABOVE 50 down to 50
mask(<50, 50)    : [45, 78, 92, 50, 67]  <- floors values BELOW 50 up to 50
```

### When to use each of the four tools
| Tool | Best for |
|---|---|
| `np.where(cond, a, b)` | Simple two-way branch, works on any array-like, returns plain ndarray |
| `np.select([conds], [choices], default=)` | Three or more mutually exclusive conditions, evaluated in order |
| `Series.where(cond, other)` | Pandas-native, keeps original index/dtype better than `np.where`, "keep if true" semantics |
| `Series.mask(cond, other)` | Pandas-native, "replace if true" semantics — the logical inverse of `where` |

### Interview Insight
> "What's the real difference between `Series.where()` and `Series.mask()`?"
They're logical inverses of each other: `where(cond, other)` keeps the original value where
`cond` is `True` and substitutes `other` where `False`; `mask(cond, other)` substitutes `other`
where `cond` is `True` and keeps the original where `False`. A common way to remember: `mask`
"masks out" (hides/replaces) whatever matches the condition.


---
# CHAPTER 21 — GroupBy

## Concept
`groupby()` implements the **split-apply-combine** pattern: split rows into groups by a key,
apply an aggregation to each group independently, combine the results into one output.
```
Split                Apply              Combine
department A -> [rows] -> mean(salary) -> one row per department
department B -> [rows] -> mean(salary) ->
department C -> [rows] -> mean(salary) ->
```
This is the single most-used Pandas operation in real analytical work — "X by Y" (revenue by
region, count by category, average by department) is always a `groupby`.


In [ ]:
employees = pd.DataFrame({
    "employee_id": range(1, 9),
    "department": ["Sales", "Eng", "Sales", "Eng", "Marketing", "Eng", "Sales", "Marketing"],
    "salary": [65000, 95000, 72000, 110000, 58000, 130000, 68000, 61000],
    "years_experience": [3, 6, 4, 9, 2, 12, 3, 4],
})

print("employees:\n", employees)
print()
print("groupby('department')['salary'].mean():\n", employees.groupby("department")["salary"].mean())
print()
print("groupby('department').size() (row count per group):\n", employees.groupby("department").size())


### Expected Output (verified on pandas 3.0.2)
```
employees:
   employee_id department  salary  years_experience
0            1      Sales   65000                 3
1            2        Eng   95000                 6
2            3      Sales   72000                 4
3            4        Eng  110000                 9
4            5  Marketing   58000                 2
5            6        Eng  130000                12
6            7      Sales   68000                 3
7            8  Marketing   61000                 4

groupby('department')['salary'].mean():
department
Eng          111666.666667
Marketing     59500.000000
Sales         68333.333333
Name: salary, dtype: float64

groupby('department').size() (row count per group):
department
Eng          3
Marketing    2
Sales        3
dtype: int64
```

### `size()` vs `count()` — a real, easy-to-confuse distinction
`.size()` counts **all rows** in each group, including rows with missing values in other
columns. `.count()` (applied per-column) counts **non-missing values** in each column, per
group — these can differ the moment any grouped column has `NaN`s. Always be explicit about
which one you mean.

## `.agg()` with named aggregations — the professional, production-quality pattern


In [ ]:
summary = employees.groupby("department").agg(
    avg_salary=("salary", "mean"),
    max_salary=("salary", "max"),
    min_salary=("salary", "min"),
    employee_count=("employee_id", "count"),
    avg_experience=("years_experience", "mean"),
)
print(summary)
print("\nsummary.index:", summary.index.tolist(), " <- department became the INDEX, not a column")
print("summary.reset_index():\n", summary.reset_index(), "\n <- pulls it back into a normal column, Chapter 49's territory")


### Expected Output (verified on pandas 3.0.2)
```
             avg_salary  max_salary  min_salary  employee_count  avg_experience
department
Eng        111666.666667      130000       95000               3        9.000000
Marketing   59500.000000       61000       58000               2        3.000000
Sales       68333.333333       72000       65000               3        3.333333

summary.index: ['Eng', 'Marketing', 'Sales']  <- department became the INDEX, not a column

summary.reset_index():
  department     avg_salary  max_salary  min_salary  employee_count  avg_experience
0        Eng  111666.666667      130000       95000               3        9.000000
1  Marketing   59500.000000       61000       58000               2        3.000000
2      Sales   68333.333333       72000       65000               3        3.333333
```

### Line-by-line: `employees.groupby("department").agg(avg_salary=("salary", "mean"), ...)`
- `employees.groupby("department")` — creates a `GroupBy` object (a deferred, not-yet-computed
  grouping — nothing is aggregated yet at this point).
- `.agg(...)` — triggers the actual split-apply-combine computation.
- `avg_salary=("salary", "mean")` — **named aggregation** syntax: the keyword (`avg_salary`)
  becomes the **output column name**; the tuple's first element (`"salary"`) is which **input
  column** to aggregate; the second element (`"mean"`) is the **aggregation function**.
- This named syntax is strongly preferred over older `.agg({"salary": "mean"})` dict-style
  syntax specifically because it lets you give the **output column a clear name directly** and
  compute **multiple different aggregations of the same input column** in one call (e.g.,
  `max_salary` and `min_salary` both aggregate `salary`, just differently) — the dict-style
  syntax struggles with this since dict keys must be unique.

### Why the grouping column becomes the **index**, and why `reset_index()` matters
`groupby().agg()` output has the grouping key(s) as the **row index**, not a regular column —
this is often surprising the first time, and matters practically because `department` won't show
up in `summary.columns`, and code expecting it as a normal column (e.g., before merging this
summary back with another table, Chapter 24) needs `.reset_index()` first to pull it back into
column form. **This is precisely why Chapter 23 (MultiIndex) comes right before Chapter 24
(Merge) in this course — grouped results routinely need `reset_index()` before they can be
joined to anything else.**

### `as_index=False` — an alternative to reset_index(), decided at groupby time


In [ ]:
summary_flat = employees.groupby("department", as_index=False).agg(
    avg_salary=("salary", "mean"),
)
print("groupby(..., as_index=False):\n", summary_flat)
print("\nEquivalent to groupby(...).agg(...).reset_index()?",
      summary_flat.equals(employees.groupby("department").agg(avg_salary=("salary", "mean")).reset_index()))


### Expected Output (verified on pandas 3.0.2)
```
groupby(..., as_index=False):
  department    avg_salary
0        Eng  111666.666667
1  Marketing   59500.000000
2      Sales   68333.333333

Equivalent to groupby(...).agg(...).reset_index()? True
```

### Interview Insight
> "Explain what `df.groupby('department').agg(avg_salary=('salary','mean'))` does, line by
> line, including the resulting index."
`groupby('department')` splits rows into groups sharing the same `department` value.
`.agg(avg_salary=('salary','mean'))` computes the mean of `salary` within each group, naming the
output column `avg_salary`. The result's row index is the distinct `department` values (not a
default integer index) — `reset_index()` or `as_index=False` at groupby time converts that back
into a normal column, which is typically needed before merging the summary with another table.


---
# CHAPTER 22 — GroupBy Advanced

## Concept
`agg()` **reduces** each group to one row. `transform()` returns a result **the same length as
the input**, broadcasting the group's aggregate back to every row in that group — essential for
feature engineering (e.g., "this employee's salary relative to their department's average").
`filter()` keeps or drops **entire groups** based on a group-level condition.


In [ ]:
employees["dept_avg_salary"] = employees.groupby("department")["salary"].transform("mean")
employees["salary_vs_dept_avg"] = employees["salary"] - employees["dept_avg_salary"]

print(employees)
print()
print("Shape check -- transform() output is SAME LENGTH as input:")
print("employees.shape:", employees.shape[0], "rows")
print("transform() result length:", len(employees.groupby('department')['salary'].transform('mean')))


### Expected Output (verified on pandas 3.0.2)
```
   employee_id department  salary  years_experience  dept_avg_salary  salary_vs_dept_avg
0            1      Sales   65000                 3     68333.333333        -3333.333333
1            2        Eng   95000                 6    111666.666667       -16666.666667
2            3      Sales   72000                 4     68333.333333         3666.666667
3            4        Eng  110000                 9    111666.666667        -1666.666667
4            5  Marketing   58000                 2     59500.000000        -1500.000000
5            6        Eng  130000                12    111666.666667        18333.333333
6            7      Sales   68000                 3     68333.333333         -333.333333
7            8  Marketing   61000                 4     59500.000000         1500.000000

Shape check -- transform() output is SAME LENGTH as input:
employees.shape: 8 rows
transform() result length: 8
```

### `agg()` vs `transform()` — the shape difference is the whole story
`agg()` on 3 departments produces **3 rows** (one per group). `transform()` on the same grouping
produces **8 rows** (one per original row) — each row gets its *own* department's aggregate
value broadcast back to it, exactly like NumPy's `keepdims=True` + broadcasting pattern from the
NumPy course, now at the DataFrame level. This is precisely why `transform()`, not `agg()`, is
the tool for creating a new **feature column** (like `salary_vs_dept_avg` above) — you need a
value aligned with every original row, not a separate small summary table.

### `filter()` — keeping/dropping whole groups based on a group-level condition


In [ ]:
# Keep only departments with at least 3 employees
large_departments = employees.groupby("department").filter(lambda g: len(g) >= 3)
print("filter(lambda g: len(g) >= 3) -- keeps/drops ENTIRE groups:\n", large_departments)
print("\nDepartments remaining:", large_departments['department'].unique().tolist())
print("(Marketing, with only 2 employees, was ENTIRELY excluded -- not just some of its rows)")


### Expected Output (verified on pandas 3.0.2)
```
filter(lambda g: len(g) >= 3) -- keeps/drops ENTIRE groups:
   employee_id department  salary  years_experience  dept_avg_salary  salary_vs_dept_avg
0            1      Sales   65000                 3     68333.333333        -3333.333333
1            2        Eng   95000                 6    111666.666667       -16666.666667
2            3      Sales   72000                 4     68333.333333         3666.666667
3            4        Eng  110000                 9    111666.666667        -1666.666667
5            6        Eng  130000                12    111666.666667        18333.333333
6            7      Sales   68000                 3     68333.333333         -333.333333

Departments remaining: ['Sales', 'Eng']
(Marketing, with only 2 employees, was ENTIRELY excluded -- not just some of its rows)
```

### The precise distinction: `agg` vs `transform` vs `filter` vs `apply`
| Method | Output shape | Purpose |
|---|---|---|
| `agg()` | One row per group (reduced) | Summary statistics, reporting tables |
| `transform()` | Same shape as input | Feature engineering — group-relative values aligned to original rows |
| `filter()` | Subset of ORIGINAL rows (whole groups kept/dropped) | Removing entire groups that fail a group-level condition (e.g., too few observations) |
| `apply()` | Arbitrary — depends entirely on what the function returns | Genuinely custom, non-standard group logic that doesn't fit the other three |

### The `groupby(dropna=False)` fix from Chapter 12's live bug, now properly explained


In [ ]:
df_with_missing_group = pd.DataFrame({
    "region": ["East", "West", None, "East", None],
    "score": [85, 90, 78, 88, 92],
})

default_behavior = df_with_missing_group.groupby("region")["score"].transform("mean")
fixed_behavior = df_with_missing_group.groupby("region", dropna=False)["score"].transform("mean")

print("Default (dropna=True) -- NaN-region rows get NaN back, ORIGINAL VALUES LOST:")
print(default_behavior.tolist())
print("\ndropna=False -- NaN itself becomes a valid group, original values PRESERVED via their own group mean:")
print(fixed_behavior.tolist())


### Expected Output (verified on pandas 3.0.2)
```
Default (dropna=True) -- NaN-region rows get NaN back, ORIGINAL VALUES LOST:
[86.5, 90.0, nan, 86.5, nan]

dropna=False -- NaN itself becomes a valid group, original values PRESERVED via their own group mean:
[86.5, 90.0, 85.0, 86.5, 85.0]
```

This is Chapter 12's dangerous gotcha, now with the actual mechanism named and the fix applied
correctly: `dropna=False` tells `groupby` to treat missing keys as their own legitimate group
(here, both `NaN`-region rows form a two-row group with mean `(78+92)/2 = 85.0`) rather than
excluding those rows from every group and silently returning `NaN` for them via `transform()`.
Notice `East`'s rows (indices 0 and 3) show `86.5` — `(85+88)/2` — in **both** outputs; this
confirms `dropna` only changes what happens to rows whose *own* group key is missing, not the
computation for any other, complete group.

### Interview Insight
> "When would you use `transform()` instead of `agg()` in a feature engineering context?"
When the new value needs to be **aligned with every original row** for use as a model feature
or for row-level comparison (e.g., "this transaction's amount relative to this customer's
average transaction amount") — `agg()` collapses to one row per group, losing that per-row
alignment entirely; `transform()` broadcasts the group statistic back to every row's original
position, giving you a same-length column ready to use directly alongside the rest of the data.


---
# CHAPTER 23 — MultiIndex

## Concept
Grouping by **multiple** columns produces a hierarchical row index (a `MultiIndex`) — one level
per grouping column. Knowing how to select from it and flatten it back is essential, since it's
the direct, unavoidable output of any multi-key `groupby`.


In [ ]:
sales = pd.DataFrame({
    "region": ["East", "East", "West", "West", "East", "West"],
    "product": ["A", "B", "A", "B", "A", "A"],
    "revenue": [1000, 1500, 1200, 900, 1100, 1300],
})

grouped = sales.groupby(["region", "product"])["revenue"].sum()
print("Multi-key groupby result (a MultiIndex Series):\n", grouped)
print("\ntype(grouped.index):", type(grouped.index).__name__)
print("grouped.index.names:", grouped.index.names)


### Expected Output (verified on pandas 3.0.2)
```
Multi-key groupby result (a MultiIndex Series):
region  product
East    A          2100
        B          1500
West    A          2500
        B           900
Name: revenue, dtype: int64

type(grouped.index): MultiIndex
grouped.index.names: FrozenList(['region', 'product'])
```

### Selecting from a MultiIndex


In [ ]:
print("grouped.loc['East']  (all products within East):\n", grouped.loc["East"])
print("\ngrouped.loc['East', 'A']  (specific region+product):", grouped.loc["East", "A"])
print("\ngrouped.loc[('West', 'B')]  (tuple form, equivalent):", grouped.loc[("West", "B")])

# reset_index() -- the standard way to flatten back into normal columns
flat = grouped.reset_index()
print("\nreset_index() -- back to a normal flat DataFrame:\n", flat)


### Expected Output (verified on pandas 3.0.2)
```
grouped.loc['East']  (all products within East):
product
A    2100
B    1500
Name: revenue, dtype: int64

grouped.loc['East', 'A']  (specific region+product): 2100

grouped.loc[('West', 'B')]  (tuple form, equivalent): 900

reset_index() -- back to a normal flat DataFrame:
  region product  revenue
0   East       A     2100
1   East       B     1500
2   West       A     2500
3   West       B      900
```

### `unstack()` — pivoting one MultiIndex level into columns (a preview of Chapter 27)


In [ ]:
unstacked = grouped.unstack()   # moves the SECOND index level ('product') into COLUMNS
print("unstack() -- 'product' level becomes columns:\n", unstacked)
print("\nThis is now a normal 2D table -- region as rows, product as columns, revenue as values")


### Expected Output (verified on pandas 3.0.2)
```
unstack() -- 'product' level becomes columns:
product     A     B
region
East     2100  1500
West     2500   900

This is now a normal 2D table -- region as rows, product as columns, revenue as values
```

### Why this course doesn't dwell on MultiIndex — a genuine, practical caveat
MultiIndex is a natural, unavoidable *byproduct* of multi-key `groupby`/`pivot_table` operations
— you will encounter it constantly. But **deliberately building complex, deeply-nested
MultiIndex structures as a primary data design choice is usually a sign of over-engineering** —
most real analytical and ML workflows are better served by flat columns (`reset_index()`
immediately after grouping) that are simpler to filter, merge, and reason about. **The
professional skill is recognizing MultiIndex when `groupby`/`pivot_table` produces it, knowing
how to select from and flatten it — not building elaborate MultiIndex hierarchies by choice.**

### Interview Insight
> "You call `df.groupby(['a','b']).sum()` and want to merge the result with another table. What
> do you need to do first?"
`reset_index()` — the grouped result has `a` and `b` as a `MultiIndex`, not regular columns, and
`merge()` (Chapter 24) needs actual columns to join on. `reset_index()` converts the index
levels back into normal DataFrame columns, after which the result can be merged like any other
flat table.


---
# CHAPTER 24 — Merge

## Concept
`pd.merge()` combines two DataFrames based on matching key column(s), following the same
conceptual model as SQL joins. This is arguably the highest-stakes operation in this entire
course to get wrong — a bad join silently produces a **wrong row count**, and unlike most bugs,
it often looks completely fine on casual inspection.

## The five join types, precisely
```
INNER  (how='inner', default): only rows with a matching key in BOTH tables
LEFT   (how='left'):            all rows from the LEFT table, matched data from right where it exists (else NaN)
RIGHT  (how='right'):           all rows from the RIGHT table, matched data from left where it exists (else NaN)
OUTER  (how='outer'):           all rows from BOTH tables, NaN where no match exists on either side
CROSS  (how='cross'):           every row of left paired with every row of right (no key needed) -- rare, deliberate use
```


In [ ]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Carol", "Dave"],
})

orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 2, 5, 1],   # note: customer_id 5 doesn't exist in `customers`; customer 3 & 4 have NO orders
    "amount": [50.0, 75.0, 30.0, 90.0, 45.0],
})

print("customers:\n", customers)
print("\norders:\n", orders)
print()
print("INNER join (default):\n", pd.merge(customers, orders, on="customer_id", how="inner"))


### Expected Output (verified on pandas 3.0.2)
```
customers:
   customer_id   name
0            1  Alice
1            2    Bob
2            3  Carol
3            4   Dave

orders:
   order_id  customer_id  amount
0       101            1    50.0
1       102            2    75.0
2       103            2    30.0
3       104            5    90.0
4       105            1    45.0

INNER join (default):
   customer_id   name  order_id  amount
0            1  Alice       101    50.0
1            1  Alice       105    45.0
2            2    Bob       102    75.0
3            2    Bob       103    30.0
```

### Reading the INNER join result precisely
Only customers **with at least one matching order** appear (Alice, Bob) — Carol and Dave (no
orders) are dropped entirely, and the order for `customer_id=5` (doesn't exist in `customers`)
is also dropped, since it has no match on the customer side. Notice Alice appears **twice** —
this is completely correct: she has two orders, and an inner join naturally produces one output
row per matching pair, so a customer with multiple orders legitimately produces multiple output
rows. **This row-multiplication behavior is the single most common source of "why did my join
inflate my row count" confusion — it's not a bug, it's the correct behavior of a one-to-many
relationship, and Chapter 25 covers how to detect when it happens unexpectedly.**


In [ ]:
print("LEFT join (all customers, even those with no orders):\n",
      pd.merge(customers, orders, on="customer_id", how="left"))

print("\nRIGHT join (all orders, even those with no matching customer):\n",
      pd.merge(customers, orders, on="customer_id", how="right"))

print("\nOUTER join (everything from both sides):\n",
      pd.merge(customers, orders, on="customer_id", how="outer"))


### Expected Output (verified on pandas 3.0.2)
```
LEFT join (all customers, even those with no orders):
   customer_id   name  order_id  amount
0            1  Alice     101.0    50.0
1            1  Alice     105.0    45.0
2            2    Bob     102.0    75.0
3            2    Bob     103.0    30.0
4            3  Carol       NaN     NaN
5            4   Dave       NaN     NaN

RIGHT join (all orders, even those with no matching customer):
   customer_id   name  order_id  amount
0            1  Alice       101    50.0
1            2    Bob       102    75.0
2            2    Bob       103    30.0
3            5    NaN       104    90.0
4            1  Alice       105    45.0

OUTER join (everything from both sides):
   customer_id   name  order_id  amount
0            1  Alice     101.0    50.0
1            1  Alice     105.0    45.0
2            2    Bob     102.0    75.0
3            2    Bob     103.0    30.0
4            3  Carol       NaN     NaN
5            4   Dave       NaN     NaN
6            5    NaN     104.0    90.0
```

### Reading `LEFT` join carefully — this is the join you'll use most often in practice
`LEFT` keeps **every** row from `customers` (the left table) regardless of whether it has any
matching orders — Carol and Dave appear with `NaN` for `order_id`/`amount`, correctly signaling
"this customer exists but has never ordered." This is the standard join for "enrich my primary
entity table with related data, but don't lose any entities that happen to have no related
records" — e.g., "all customers, with their total spend if any" for a churn analysis where
"has never ordered" is itself meaningful information you don't want silently dropped by an
inner join.

### `left_on`/`right_on` — when the key columns have different names


In [ ]:
products = pd.DataFrame({"product_code": ["A1", "B2", "C3"], "product_name": ["Widget", "Gadget", "Gizmo"]})
sales_records = pd.DataFrame({"item_code": ["A1", "B2", "A1"], "units_sold": [10, 5, 8]})

merged = pd.merge(products, sales_records, left_on="product_code", right_on="item_code", how="inner")
print("merge(left_on='product_code', right_on='item_code'):\n", merged)
print("\n<- Notice BOTH key columns survive in the output (product_code AND item_code) --")
print("   drop the redundant one afterward if you don't need it: merged.drop(columns='item_code')")


### Expected Output (verified on pandas 3.0.2)
```
merge(left_on='product_code', right_on='item_code'):
  product_code product_name item_code  units_sold
0           A1       Widget        A1          10
1           A1       Widget        A1           8
2           B2       Gadget        B2           5

<- Notice BOTH key columns survive in the output (product_code AND item_code) --
   drop the redundant one afterward if you don't need it: merged.drop(columns='item_code')
```

### `suffixes` — resolving column name collisions


In [ ]:
left_df = pd.DataFrame({"id": [1, 2], "value": [100, 200]})
right_df = pd.DataFrame({"id": [1, 2], "value": [10, 20]})

default_suffix = pd.merge(left_df, right_df, on="id")
custom_suffix = pd.merge(left_df, right_df, on="id", suffixes=("_before", "_after"))

print("Default suffixes ('_x','_y'):\n", default_suffix)
print("\nCustom suffixes:\n", custom_suffix)


### Expected Output (verified on pandas 3.0.2)
```
Default suffixes ('_x','_y'):
   id  value_x  value_y
0   1      100       10
1   2      200       20

Custom suffixes:
   id  value_before  value_after
0   1           100           10
1   2           200           20
```

### `validate` — the parameter every professional merge should use


In [ ]:
try:
    pd.merge(customers, orders, on="customer_id", how="inner", validate="one_to_one")
except pd.errors.MergeError as e:
    print("validate='one_to_one' correctly RAISED an error, because it's actually one-to-many:")
    print(" ", e)

# The CORRECT validate value for this actual relationship:
result = pd.merge(customers, orders, on="customer_id", how="inner", validate="one_to_many")
print("\nvalidate='one_to_many' (the TRUE relationship here) succeeds:\n", result)


### Expected Output (verified on pandas 3.0.2)
```
validate='one_to_one' correctly RAISED an error, because it's actually one-to-many:
  Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
  customer_id
           2
           1 ...

validate='one_to_many' (the TRUE relationship here) succeeds:
   customer_id   name  order_id  amount
0            1  Alice       101    50.0
1            1  Alice       105    45.0
2            2    Bob       102    75.0
3            2    Bob       103    30.0
```

Note pandas 3.0 made this error message even more useful than earlier versions — it now shows
the actual duplicate keys it found (`customer_id` values `2` and `1` each appear more than
once in the right table), not just the fact that duplication occurred. This makes
`validate=`'s failures directly actionable without any extra debugging step.

### Why `validate=` is the single highest-leverage parameter in this entire chapter
`validate` doesn't change the join's result — it adds an **assertion** that the merge key
relationship matches what you *expect* it to be, raising a clear error immediately if it
doesn't. This is exactly the defensive-programming instinct from the NumPy course (shape
assertions, `np.testing`) applied to joins: **if you believe a merge should be one-to-one (e.g.,
joining two tables that should each have exactly one row per customer), `validate='one_to_one'`
will catch it immediately and loudly if that assumption is wrong** — rather than silently
producing a row-inflated result that might not be discovered until a much later, harder-to-trace
stage of a pipeline (Chapter 25 covers exactly this failure mode). **Making `validate=` a
default habit, not an occasional add-on, is one of the highest-leverage professional practices
in this entire course.**

### Interview Insight
> "Why would you add `validate='one_to_one'` (or another validate value) to a merge, given it
> doesn't change the actual join result when the assumption holds?"
It converts a silent, potentially-wrong assumption about the data's key relationship into an
explicit, loudly-failing assertion — exactly like a unit test for the merge itself. If the
relationship is genuinely one-to-one and stays that way, `validate` costs nothing extra; if it
silently becomes one-to-many (e.g., due to an upstream data quality regression introducing
duplicate keys), `validate` catches it immediately at the merge, rather than allowing a silently
row-inflated result to propagate through the rest of a pipeline undetected.


---
## Chapters 19–24 Summary

- **Vectorization**: `.iloc[i]` inside a row-loop is a severe, common anti-pattern — verified
  thousands of times slower than direct column arithmetic; NumPy ufuncs work directly on Series.
- **Conditional columns**: `np.where` (binary), `np.select` (multi-way, ordered, first-match),
  `Series.where`/`Series.mask` (Pandas-native, logical inverses of each other — "keep if" vs
  "replace if").
- **GroupBy**: split-apply-combine; named aggregation (`agg(name=("col","func"))`) is the
  professional pattern; the grouping key becomes the **index**, not a column — `reset_index()`
  or `as_index=False` needed before merging elsewhere.
- **GroupBy advanced**: `agg()` reduces to one row per group; `transform()` broadcasts back to
  original row count (the feature-engineering tool); `filter()` keeps/drops entire groups;
  `groupby(dropna=False)` fixes the real, dangerous NaN-group-key data-loss bug demonstrated
  live in Chapter 12 and formalized here.
- **MultiIndex**: the natural, unavoidable output of multi-key grouping — know how to
  `.loc[]` into it and `reset_index()`/`unstack()` it back to flat form; avoid *designing* deep
  MultiIndex hierarchies deliberately, since flat columns are almost always easier to work with.
- **Merge**: inner/left/right/outer, precisely defined and verified with real row counts;
  row-count inflation from one-to-many relationships is correct behavior, not a bug — but
  `validate=` should be a default habit to catch it when it's *unexpected*, exactly the
  defensive-assertion discipline carried over from the NumPy course.

---

## Mastery Exercises

**Exercise 1 (Prediction):** For `customers` (4 rows, all unique `customer_id`) and `orders` (5
rows, `customer_id` values `[1,2,2,5,1]`), predict the row count of an INNER join, a LEFT join,
and an OUTER join on `customer_id` — before checking against this chapter's actual output.

**Exercise 2:** Given `employees.groupby("department")["salary"].transform("mean")`, explain in
one sentence why this returns 8 values (matching the original row count) rather than 3 (matching
the number of departments).

**Coding Challenge:** Write `safe_merge(left, right, on, expected_relationship)` that wraps
`pd.merge()` with the given `validate=expected_relationship`, catches `pd.errors.MergeError`,
and re-raises a more informative exception including the actual row counts of `left`, `right`,
and what the join would have produced without validation (compute this separately) — so a
caller immediately sees both what was expected and what the data actually contains.

**Debugging Challenge:**
```python
customers_df = pd.DataFrame({"id": [1,2,3], "name": ["A","B","C"]})
transactions_df = pd.DataFrame({"id": [1,1,2,2,3,3], "amount": [10,20,30,40,50,60]})
result = pd.merge(customers_df, transactions_df, on="id")
print(len(result))   # a teammate expected 3 rows (matching customers_df), got 6
```
Explain exactly why, referencing this chapter's row-multiplication lesson, and state which
`validate` value would have caught this if the teammate had genuinely expected one-to-one.

**Interview Questions:**
1. Precisely define the difference between LEFT and OUTER joins.
2. Why does a grouped `.agg()` result have the grouping column as its index rather than a
   regular column, and what's the standard fix before merging it elsewhere?
3. Give a concrete feature-engineering example where `transform()` is the right tool and
   `agg()` is not.
4. What does `validate='one_to_many'` actually check, and why is it good practice to include
   even when you're confident about the relationship?

**Quiz:**
- Q1: True/False — An inner join can produce MORE rows than either input table.
- Q2: Which `groupby` parameter treats missing group keys as their own valid group instead of
  excluding those rows entirely?
- Q3: For `pd.merge(a, b, on='id')` where `id` is unique in `a` but has duplicates in `b`,
  which `validate` value correctly describes this relationship?

---
 